In [4]:
import sys
print(f"正在為這個 Python 安裝套件: {sys.executable}")

# 強制使用當前核心的 pip 進行安裝
!{sys.executable} -m pip install newspaper3k lxml_html_clean

正在為這個 Python 安裝套件: /opt/anaconda3/bin/python


In [1]:
try:
    import newspaper
    from newspaper import Article
    print("✅ 成功！現在 Python 已經抓得到 newspaper3k 了！")
    print(f"安裝位置: {newspaper.__file__}")
except ImportError as e:
    print(f"❌ 還是失敗：{e}")

✅ 成功！現在 Python 已經抓得到 newspaper3k 了！
安裝位置: /opt/anaconda3/lib/python3.13/site-packages/newspaper/__init__.py


In [2]:
# Cell 1: 匯入套件與設定參數
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timezone, timedelta
import os
import time
import random

# --- 設定偽裝 Header ---
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://news.google.com/'
}

# --- 設定欄位名稱 (新增 category) ---
COLUMNS = [
    'category',       # <--- 新增：主題類別
    'url', 
    'title', 
    'content', 
    'source',
    'published_at', 
    'scraped_at', 
    'headline_level'
]

OUTPUT_FILENAME = "news_datasets_googlenews.csv"
print("✅ 環境設定完成，已新增 'category' 欄位！")

✅ 環境設定完成，已新增 'category' 欄位！


In [3]:
# Cell 2: 定義主題網址字典
# 這些是 Google News 台灣版的標準分類連結 ID

TOPIC_URLS = {
    "焦點頭條": "https://news.google.com/topics/CAAqKggKIiRDQkFTRlFvSUwyMHZNRFZxYUdjU0JXVnVMVWRDR2dKSlRpZ0FQAQ?hl=zh-TW&gl=TW&ceid=TW%3Azh-Hant",
    #"台灣在地": "https://news.google.com/topics/CAAqKggKIiRDQkFTRlFvSUwyMHZNRFZxYUdjU0JXVnVMVWRDR2dKSlRpZ0FQAQ/sections/CAQiR0NCQVNMd29JTDIwdk1ERnpxYUdjU0JXVnVMVWRDR2dKSlRpZ0tDVWxEQW9JQWtjQ0FqZ2tDQW9VRQoyC3RvcGljcy9kYXRhKABQAVAB?hl=zh-TW&gl=TW&ceid=TW%3Azh-Hant",
    "國際財經": "https://news.google.com/topics/CAAqKggKIiRDQkFTRlFvSUwyMHZNRGx6TVdZU0JXVnVMVWRDR2dKSlRpZ0FQAQ?hl=zh-TW&gl=TW&ceid=TW%3Azh-Hant",
    "科技科學": "https://news.google.com/topics/CAAqKggKIiRDQkFTRlFvSUwyMHZNRGRqTVhZU0JXVnVMVWRDR2dKSlRpZ0FQAQ?hl=zh-TW&gl=TW&ceid=TW%3Azh-Hant",
    "運動體育": "https://news.google.com/topics/CAAqKggKIiRDQkFTRlFvSUwyMHZNRFp1ZEdvU0JXVnVMVWRDR2dKSlRpZ0FQAQ?hl=zh-TW&gl=TW&ceid=TW%3Azh-Hant"
}

print(f"✅ 已設定 {len(TOPIC_URLS)} 個抓取目標：{list(TOPIC_URLS.keys())}")

✅ 已設定 4 個抓取目標：['焦點頭條', '國際財經', '科技科學', '運動體育']


In [4]:
# Cell 3: 定義輔助函式
def get_current_time_str():
    taiwan_tz = timezone(timedelta(hours=8))
    return datetime.now(taiwan_tz).strftime("%Y-%m-%d %H:%M:%S")

def clean_google_link(link):
    if link.startswith("./"):
        return "https://news.google.com" + link[1:]
    return link

print("✅ 輔助函式準備就緒！")

✅ 輔助函式準備就緒！


In [7]:
# Cell 4: 執行多主題抓取 (中轉頁面爬取版 + 內文抓取)

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# --- 1. 設定 newspaper ---
try:
    from newspaper import Article, Config
    HAS_NEWSPAPER = True
    news_config = Config()
    news_config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    news_config.request_timeout = 10
except ImportError:
    HAS_NEWSPAPER = False

# --- 2. 定義中轉頁面解析器 (救援核心) ---
def get_redirect_url(google_url):
    """
    連線到 Google 的轉導頁面，從 HTML 中找出真實的跳轉連結。
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            # 加入 Cookie 讓 Google 認為我們是人類，有助於取得正確轉導頁
            'Cookie': 'CONSENT=YES+Cb.20210328-17-p0.zh-CN+FX+417; PREF=ID=1111111111111111:FF=0:LD=zh-TW:TM=1617000000:LM=1637000000:S=XxXxXxXxXxXxXxXx' 
        }
        # 請求該頁面
        response = requests.get(google_url, headers=headers, timeout=8)
        
        # 方法 A: 檢查是否已經自動轉址成功
        if 'google.com' not in response.url:
            return response.url
            
        # 方法 B: 如果還在 google，分析 HTML 找出目標連結
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Google 的轉導頁通常會有一個主要的 <a href="..."> 讓使用者點擊
        # 我們尋找頁面上所有的連結，找出第一個「非 Google」的外部連結
        links = soup.find_all('a', href=True)
        for link in links:
            href = link['href']
            # 如果連結是完整的 http 開頭，且不是連回 google 自身的
            if href.startswith('http') and 'google.com' not in href and 'google.co' not in href:
                return href
                
        return None
    except Exception:
        return None

# --- 3. 主程式 ---
all_data_list = []
current_time = get_current_time_str()

print(f"🚀 開始執行爬蟲 (啟用中轉頁面解析)...")
print(f"⚠️ 因為需要連線解析網址，速度會較慢，請耐心等候 (每篇約需 2-3 秒)")

for category_name, target_url in TOPIC_URLS.items():
    if category_name == "台灣在地": continue
        
    print(f"\n📡 正在抓取類別：【{category_name}】...")
    
    try:
        # 取得列表頁
        response = requests.get(target_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(response.text, 'html.parser')
        articles = soup.find_all('article')
        print(f"   ↳ 找到 {len(articles)} 篇新聞區塊")

        count = 0
        # --- 為了測試，建議先維持前 5 篇，確認有抓到內文再全跑 ---
        # --- 若要全跑，請移除 [:5] ---
        for index, article in enumerate(articles[:5]): 
            try:
                # A. 抓取標題與 Google 連結
                all_links = article.find_all('a', href=True)
                candidate_links = []
                for link in all_links:
                    href = link['href']
                    if ('./read/' in href) or ('./articles/' in href):
                        text = link.get_text(strip=True)
                        candidate_links.append({'text': text, 'len': len(text), 'href': href, 'element': link})
                
                if not candidate_links: continue
                candidate_links.sort(key=lambda x: x['len'], reverse=True)
                best_candidate = candidate_links[0]
                
                title = best_candidate['text'] or best_candidate['element'].get('aria-label') or "無標題"
                google_url = clean_google_link(best_candidate['href'])
                
                # B. 來源與時間
                source_div = article.find('div', class_='vr1PYe')
                source = source_div.get_text(strip=True) if source_div else "Google News"
                time_tag = article.find('time')
                published_at = time_tag['datetime'].replace('T', ' ').replace('Z', '') if (time_tag and time_tag.has_attr('datetime')) else current_time
                
                # C. 層級
                headline_level = 3 if (category_name == "焦點頭條" and index < 5) else 2

                # --- D. 關鍵：解析真實網址 ---
                real_url = google_url
                content = "N/A"
                
                # 1. 呼叫解析函式
                decoded_url = get_redirect_url(google_url)
                
                if decoded_url:
                    real_url = decoded_url
                    
                    # 2. 只有解析成功才抓內文
                    if HAS_NEWSPAPER:
                        try:
                            article_obj = Article(real_url, config=news_config)
                            article_obj.download()
                            article_obj.parse()
                            if article_obj.text and len(article_obj.text) > 50:
                                content = article_obj.text.strip()
                            else:
                                content = "Content too short"
                        except Exception as e:
                            content = "Download Error"
                else:
                    content = "Failed to resolve URL"

                # E. 儲存
                all_data_list.append({
                    'category': category_name,
                    'url': real_url,
                    'title': title,
                    'content': content,
                    'source': source,
                    'published_at': published_at,
                    'scraped_at': current_time,
                    'headline_level': headline_level
                })
                count += 1
                
                # 進度顯示
                status_icon = "✅" if len(content) > 50 else "⚠️"
                domain = real_url.split('/')[2] if 'http' in real_url else 'google.com'
                print(f"      [{count}] {source} -> {domain} ({status_icon})")
                # 稍微休息一下，避免對目標網站造成負擔
                time.sleep(1)

            except Exception as e:
                continue
        
        print(f"   ✅ {category_name} 完成，共收集 {count} 篇")

    except Exception as e:
        print(f"   🔥 {category_name} 抓取失敗: {e}")

print(f"\n🎉 全部任務結束！總共收集到 {len(all_data_list)} 筆資料。")

🚀 開始執行爬蟲 (啟用中轉頁面解析)...
⚠️ 因為需要連線解析網址，速度會較慢，請耐心等候 (每篇約需 2-3 秒)

📡 正在抓取類別：【焦點頭條】...
   ↳ 找到 264 篇新聞區塊
      [1] 日經中文網 -> news.google.com (⚠️)
      [2] 奇摩新聞 -> news.google.com (⚠️)
      [3] 奇摩新聞 -> news.google.com (⚠️)
      [4] 轉角國際udn Global -> news.google.com (⚠️)
      [5] ETtoday新聞雲 -> news.google.com (⚠️)
   ✅ 焦點頭條 完成，共收集 5 篇

📡 正在抓取類別：【國際財經】...
   ↳ 找到 274 篇新聞區塊
      [1] Yahoo -> news.google.com (⚠️)
      [2] 聯合新聞網 -> news.google.com (⚠️)
      [3] NOWnews今日新聞 -> news.google.com (⚠️)
      [4] 中央社 CNA -> news.google.com (⚠️)
      [5] news.cnyes.com -> news.google.com (⚠️)
   ✅ 國際財經 完成，共收集 5 篇

📡 正在抓取類別：【科技科學】...
   ↳ 找到 187 篇新聞區塊
      [1] 奇摩新聞 -> news.google.com (⚠️)
      [2] 聯合新聞網 -> news.google.com (⚠️)
      [3] 自由財經 -> news.google.com (⚠️)
      [4] 台視新聞網 -> news.google.com (⚠️)
      [5] 奇摩新聞 -> news.google.com (⚠️)
   ✅ 科技科學 完成，共收集 5 篇

📡 正在抓取類別：【運動體育】...
   ↳ 找到 209 篇新聞區塊
      [1] 聯合新聞網 -> news.google.com (⚠️)
      [2] 奇摩運動 -> news.google.com (⚠️)
      [3] 自由體育 -> 

In [ ]:
# Cell 5: 最終存檔與內文品質檢查

if all_data_list:
    # 1. 轉換為 DataFrame
    df_new = pd.DataFrame(all_data_list, columns=COLUMNS)
    
    # 2. 內文品質檢查 (Content Quality Check)
    # 計算有多少篇是抓取失敗的 (例如 N/A 或 Error)
    def check_content_status(text):
        if not text or text.startswith("N/A") or text.startswith("Error") or text == "Content too short or failed to parse":
            return "Failed"
        return "Success"

    df_new['content_status'] = df_new['content'].apply(check_content_status)
    
    print("\n📊 === 最終資料品質報告 ===")
    print(f"1. 總筆數: {len(df_new)}")
    print(f"2. 內文抓取成功率:\n{df_new['content_status'].value_counts(normalize=True) * 100}")
    
    print("\n3. 成功抓到內文的範例 (前 100 字):")
    success_df = df_new[df_new['content_status'] == 'Success']
    if not success_df.empty:
        print(f"   標題: {success_df.iloc[0]['title']}")
        print(f"   內文: {success_df.iloc[0]['content'][:100]}...")
    else:
        print("   (無成功範例)")

    # 3. 執行存檔
    file_exists = os.path.exists(OUTPUT_FILENAME)
    try:
        # 存檔時不包含暫時的 check 欄位
        df_to_save = df_new.drop(columns=['content_status'])
        
        df_to_save.to_csv(
            OUTPUT_FILENAME, 
            mode='a', 
            header=not file_exists, 
            index=False, 
            encoding='utf-8-sig'
        )
        print(f"\n💾 成功寫入 {OUTPUT_FILENAME}！")
        print(f"   目前檔案大小: {os.path.getsize(OUTPUT_FILENAME) / 1024:.2f} KB")
        
    except Exception as e:
        print(f"🔥 存檔失敗: {e}")
else:
    print("⚠️ 本次未抓取到任何資料，無法存檔。")